## RL Methods: Policy Gradient

### Idea

- In **Policy Gradient** approach, we attempt to parameterize the policy $\pi$ using some set of parameters $\theta$. 

- Then, to improve our policy, we iteratively modify $\theta$ by using the first derivative of the expected reward $J_{\pi_{\theta}}$ w.r.t a change in $\theta$. That is:
\begin{aligned}
    \frac{\partial J_{\pi_{\theta}}}{\partial \theta}
\end{aligned}

- This is known as **gradient ascent**. It's an ascent, because we want to maximise the reward $J$, instead of the usual gradient descent in supervised learning when we are minimising some loss $L$

- For each iteration, we update $\theta$ by taking

\begin{aligned}
    \theta \leftarrow \theta + \alpha \cdot \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) \cdot G_t
\end{aligned}

### Theory: How does the gradient ascent update come about?

- In the policy gradient approach, we want to maximise the expected return $J$ under some policy $\pi_{\theta}$, where the policy is entirely dependent on parameter $\theta$

\begin{aligned}
    J(\theta) &= E_{\tau \sim \pi_{\theta}}[R(\tau)] \\
    &\text{where} \\
    &\quad \tau = [(s_0, a_0), (s_1, a_1), ...] \text{ is the trajectory of states and actions } (s_t, a_t) \\
    &\quad R(\tau) = \sum_{t=0}^{\infty} \gamma^t r(s_t, a_t) \text{ is the discounted total reward} \\
    &\quad \pi_{\theta}(a|s) \text{ is the theta-parameterised probability of taking action } a \text{ at state } s\\
\end{aligned}

- So if we want to take different actions, we need to modify $\theta$. And to decide how to modify $\theta$, we use the gradient $\frac{\partial J(\theta)}{\partial \theta}$. This will be written as $\bigtriangledown_{\theta} J(\theta)$

- Intuitively, since we want to maximise $J$, if the gradient is positive, we want to move in the direction of the gradient!

#### Computing the Gradient

- Let's rewrite the return $J(\theta)$ as the summation over all trajectories $\tau$, instead of leaving it as the expectation

\begin{aligned}
    J(\theta) &= \sum_{\tau} P(\tau; \theta) R(\tau)
\end{aligned}

- We know that $P(\tau; \theta)$ can be rewritten as the product of the Markov Process (i.e. the actions and transition probabilities)

\begin{aligned}
    P(\tau; \theta) &= \rho(s_0) \prod_{t=0}^{T-1} \pi_{\theta}(a_t | s_t) P(s_{t+1} | s_t, a_t) \\
    &\text{where} \\
    &\quad \rho(s_0) = \text{ Probability of starting at state } s_0 \\
    &\quad \pi_{\theta}(a_t | s_t) = \text{ Probability of taking action } a \text{ at state } s 
    \text{ under policy } \pi_{\theta} \\
    &\quad P(s_{t+1} | s_t, a_t) = \text{ Transition probability } \\
\end{aligned}

- Therefore, $\bigtriangledown_{\theta} J(\theta)$ can be written as

\begin{aligned}
    \bigtriangledown_{\theta} J(\theta) &= \bigtriangledown_{\theta} \sum_{\tau} P(\tau; \theta) R(\tau) \\
    &= \sum_{\tau} \bigtriangledown_{\theta} P(\tau, \theta) R(\tau)
\end{aligned}

- By the **log-derivative trick**, we know that:

\begin{aligned}
    \bigtriangledown_{\theta} \log f(\theta) &= \frac{\bigtriangledown_{\theta} f(\theta)}{f(\theta)} \\
    \therefore f(\theta) &= f(\theta) \bigtriangledown_{\theta} \log f(\theta)
\end{aligned}

- Therefore, it must be true that

\begin{aligned}
    \bigtriangledown_{\theta} J(\theta) &= \bigtriangledown_{\theta} \sum_{\tau} P(\tau; \theta) R(\tau) \\
    &= \sum_{\tau} \bigtriangledown_{\theta} P(\tau, \theta) R(\tau) \\
    &= \sum_{\tau} P(\tau, \theta) \bigtriangledown_{\theta} \log P(\tau, \theta) R(\tau) \\
    &= \mathbb{E}_{\tau \sim P(\tau, \theta)}[R(\tau) \bigtriangledown_{\theta} \log P(\tau, \theta)]
\end{aligned}

- This suggests that the gradient of the reward $\bigtriangledown_{\theta}J(\theta)$ is simply the expected product of reward from following trajectory $R(\tau)$, multiplied by the derivative of log probability

- Ok, but how do we compute $\bigtriangledown_{\theta} \log P(\tau, \theta)$?

\begin{aligned}
    P(\tau; \theta) &= \rho(s_0) \prod_{t=0}^{T-1} \pi_{\theta}(a_t | s_t) P(s_{t+1} | s_t, a_t) \\
    \log P(\tau; \theta) &= \log [\rho(s_0) \prod_{t=0}^{T-1} \pi_{\theta}(a_t | s_t) P(s_{t+1} | s_t, a_t)] \\
    \log P(\tau; \theta) &= \log \rho(s_0) + \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) + \sum_{t=0}^{T-1} \log P(s_{t+1} | s_t, a_t) \\
\end{aligned}

- But notice that of all the terms in the last expression, only $\sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)$ contains $\theta$! Therefore;

\begin{aligned}
    \bigtriangledown_{\theta} \log P(\tau; \theta) &= \bigtriangledown_{\theta} [\log \rho(s_0) + \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) + \sum_{t=0}^{T-1} \log P(s_{t+1} | s_t, a_t)] \\
    &= \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) \\
\end{aligned}



- Substituting back into our original gradient expression

\begin{aligned}
    \bigtriangledown_{\theta} J(\theta) &= \mathbb{E}_{\tau \sim P(\tau, \theta)}[R(\tau) \bigtriangledown_{\theta} \log P(\tau, \theta)] \\
    &= \mathbb{E}_{\tau \sim P(\tau, \theta)}[R(\tau) \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)]
\end{aligned}

- We usually subsitute the reward $R(\tau)$ with a manageable number of discounted reward steps, giving us the final expression:

\begin{aligned}
    \bigtriangledown_{\theta} J(\theta) &= \mathbb{E}_{\tau \sim P(\tau, \theta)}[R(\tau) \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)] \\
    &= \mathbb{E}_{\tau \sim P(\tau, \theta)}[\sum_{i=0}^{k} \gamma^i r_{t+i} \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)] \\
    &= \mathbb{E}_{\tau \sim P(\tau, \theta)} [G_t \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)]
\end{aligned}

- This is why, for each iteration, we update $\theta$ by taking

\begin{aligned}
    \theta \leftarrow \theta + \alpha \cdot \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) \cdot G_t
\end{aligned}

- Intuitively, the gradient moves us in the direction of improving $J(\theta)$, with $\alpha$ as the learning rate controlling step size

- Note that here, assuming an episodic task, $G_t$ is simply the discounted reward of all steps in the episode remaining after time $t$

### Theory: Choosing a differentiable policy $\pi_{\theta}$

- We established the update step above as 

\begin{aligned}
    \theta \leftarrow \theta + \alpha \cdot \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) \cdot G_t
\end{aligned}

- So the core idea here is: for any probability distribution $\pi$ across our set of actions $A = (a_0, a_1 ... a_n)$, we need it to be **differentiable w.r.t its parameters $\theta$**, because you need to compute $\bigtriangledown_{\theta} \log \pi_{\theta}(a | s)$

- That is, $\pi = [P(a_0), P(a_1) ...]$ must be a function of some $\theta$

- Here, we will go through a few examples of what forms $P(a)$ can take, so that we can differentiate the policies for both **continuous** and **discrete** action spaces 

#### Softmax Policy

- If actions are discrete, the most straightforward formulation of $\pi$ such that it is differentiable w.r.t $\theta$ is the softmax. 

- Recall that softmax has the form

\begin{aligned}
    \pi_{\theta}(a_i | s) &= \frac{e^{z_i(s, \theta) / \tau}}{\sum_j e^{z_j(s, \theta) / \tau}} \\

    &\text{where} \\
    &\quad z_j(s, \theta) = \text{ Logits output by some model }
\end{aligned}

- For a basic differentiable example, let's assume that the logits $z_i(s, \theta)$ can be treated as an OLS:

\begin{aligned}
    z_i(s, \theta) &= \theta_i^Tx
\end{aligned}

- Then, the softmax becomes

\begin{aligned}
    \pi_\theta(a_i | s) &= \frac{e^{\theta_i^Tx}}{\sum_j e^{\theta_j^Tx}} \\

    \log \pi_\theta(a_i | s) &= \log \frac{e^{\theta_i^Tx}}{\sum_j e^{\theta_j^Tx}} \\
    &= \log e^{\theta_i^Tx} - \log \sum_j e^{\theta_j^Tx} \\
    &= \theta_i^Tx - \log \sum_j e^{\theta_j^Tx} \\
\end{aligned}

- Now, let's take the derivative of $\log \pi_\theta(a_i | s)$ w.r.t $\theta_{k}$ for some value of $k$ 

- Let's clear about what this means: we know that each action $a_i$ is parameterized by some set of parameters $\theta_{i}$. Here, we want to know how $\pi_{\theta}(a_i | s)$ will change given a change in its own parameter set $\theta_i$, or some other action's parameter set $\theta_{k \neq i}$

\begin{aligned}
    \frac{\partial}{\partial \theta_k} \log \pi_(a_i | s) &= \frac{\partial}{\partial \theta_k} [\theta_i^Tx - \log \sum_j e^{\theta_j^Tx}] \\ \\    

    \frac{\partial}{\partial \theta_k} \theta_i^Tx &= \begin{cases}
        x & \text{if } k = i \\
        0 & \text{otherwise}
    \end{cases} \\
    &= x \cdot \mathbf{1}[k = i] & \text{First term} \\ \\

    \frac{\partial}{\partial \theta_k} -\log \sum_j e^{\theta_j^Tx} &= -\frac{1}{\sum_j e^{\theta_j^Tx}} e^{\theta_k^Tx} x \\
    &= -\frac{e^{\theta_k^Tx}}{\sum_j e^{\theta_j^Tx}} \cdot x \\
    &= -\pi_\theta(a = k | s) \cdot x & \text{Second term} \\ \\

    \frac{\partial}{\partial \theta_k} \log \pi_(a_i | s) &= x \cdot \mathbf{1}[k = i] - \pi_\theta(a = k | s) \cdot x \\
    &= x \cdot [\mathbf{1}[k = i] - \pi_\theta(a = k | s)] \\
    &= \begin{cases}
        x \cdot [1 - \pi_\theta(a = k | s)] & \text{if } k = i \\
        x \cdot - \pi_\theta(a = k | s) & \text{otherwise }
    \end{cases} & \text{ Combining terms }
\end{aligned}


- Let's try to put the mathematics notation into practical pseudocode

    - Let's assume we have an input $x$ of dimension $k \times b$. 
        - $b$ is the input batch size, and $k$ is the number of features in the input array

    - We previously defined the softmax logits as as linear regressor $z(s, \theta) = \theta_k^Tx$

    - Since softmax outputs a probability of length $|A|$ (i.e. one probability per action), we know that $z(s, \theta) = \theta_k^Tx$ must be of size $a \times b$\
        - $a$ is the number of actions, and $b$ is the batch size

    - Therefore, the weight matrix $W = \theta_k^T$ must have shape $a \times k$ - for every action $a$, there are $k$ parameters

    - Apply softmax to every column of the $a \times b$ matrix. That is, we convert the logits of every observation $b$ to probabilities. Therefore, probability is also an $a \times b$ matrix

    - For every observation $b$, we have picked an action. Let's record the actions chosen as $A_{\text{chosen}}$ 

    - For $j$ in observations, for $a$ in actions
        - Extract the current softmax probability of action $a$ for observation $j$. We'll call it $p[a, j]$
        - If action $a$ was chosen for observation $j$ in $A_{\text{chosen}}$, compute $1 - p[a,j]$. Otherwise, just return $p[a,j]$
        - Subset input matrix $x$ to observation $j$ by taking $x[:, j]$
        - $x[:, j]$ has shape $k \times 1$
        - $p[a, j]$ is a scalar
        - Add the product $x[:, j] * p[a, j]$ to the existing gradient estimate of action $a$

- With the term $\frac{\partial}{\partial \theta_k} \log \pi(a_i | s)$, we can now update

\begin{aligned}
    \theta \leftarrow \theta + \alpha \cdot \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) \cdot G_t
\end{aligned}

#### Gaussian Policy with Direct Parameter Update

#### Gaussian Policy with Neural Network

#### Squashed Gaussian / Tanh-Gaussian Policy

- For bounded continuous actions, you can sample a Gaussian and squash with a tanh:

#### Distributions other than Gaussian/Softmax

- Continuous
    - Beta: $\alpha_{\theta}(s), \beta_{\theta}(s)$
        - Naturally bounded (0,1); can scale to arbitrary ranges

    - Laplace / Logistic: Mean, scale
        - heavy-tailed action distributions

    - Gaussian Mixture Model: Multiple gaussians + their mixture weights
        - Multimodal actions; useful when one Gaussian cannot model the distribution

- Discrete
    - Gumbel-Softmax: logits + Gumbel noise + temperature
        - Used when you need the sampling process itself to be differentiable (i.e. not just argmax of the action-values)
        - Rare in RL

- Remember the core idea: any distribution that is **differentiable w.r.t its parameters** works, because you need to compute $\bigtriangledown_{\theta} \log \pi_{\theta}(a | s)$
    - That is, $\log P(a)$ must be a function of $\theta$, so we can compute $\frac{\partial \log P(a)}{\partial \theta}$

### Implementation

#### Setup

In [ ]:
import numpy as np
np.random.seed(0)

N_STATES = 10
N_ACTIONS = 5
GAMMA = 0.9
ALPHA = 0.1
EPISODES = 1000
TD_LAMBDA = 0.5

N_STEPS = int(1e5)

TRANSITION_PROBABILITIES = np.random.rand(N_STATES, N_ACTIONS, N_STATES)
TRANSITION_PROBABILITIES /= np.sum(TRANSITION_PROBABILITIES, axis=2, keepdims=True)
REWARDS = np.random.rand(N_STATES, N_ACTIONS, N_STATES) * 5

- First, let's implement a toy environment that will be common to all policy gradient approaches

- The entire point of the environment is to provide some sort of interaction with your actions and states, and to reset itself when needed 

- Therefore, the key method here is `def step()`, which defines how the environment transitions the actors between states, the rewards that it returns, and any other info that may be necessary

In [ ]:
class Env:
    def __init__(self):
        self.n_states = N_STATES
        self.n_actions = N_ACTIONS
        self.transition_probs = TRANSITION_PROBABILITIES
        self.rewards = REWARDS

    def reset(self):
        self.curr_state = np.random.choice(N_STATES)
        return self.curr_state
    
    def step(self, action):
        curr_state = self.curr_state
        probs = self.transition_probs[curr_state, action]
        next_state = np.random.choice(self.n_states, p=probs)

        reward = self.rewards[curr_state, action, next_state]
        self.curr_state = next_state

        done = False
        info = {}
        return next_state, reward, done, info

- Using this environment, we want to perform a pure policy gradient based update
    - the gradient term $\bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)$
    - the return $G_t$
        
    \begin{aligned}
        \theta &\leftarrow \theta + \alpha \cdot \bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t) \cdot G_t
    \end{aligned}

- Looking at the pure policy gradient approach, $G_t$ is computed using the Monte Carlo return from each timestep $t$

\begin{aligned}
    G_t &= \sum_{k=t}^{T-1} \gamma^{k-t} r_k
\end{aligned}

- Therefore, we know that the update is going to be largely identical across all approaches used below; the only difference is in the derivative term $\bigtriangledown_{\theta} \sum_{t=0}^{T-1} \log \pi_{\theta}(a_t | s_t)$

- Let's introduce a method to standardise the computation of the Monte Carlo retur

In [ ]:
def monte_carlo_return(array_of_rewards: np.ndarray) -> float:
    reward_accumulator = 0
    returns = []
    for reward in reversed(array_of_rewards):
        reward_accumulator = GAMMA * reward_accumulator + reward
        returns.append(reward_accumulator)
    
    return np.array(returns[::-1])

monte_carlo_return([1,2,3])

#### Softmax

- The softmax update step has the gradient term (See theory section above)

\begin{aligned}
\frac{\partial}{\partial \theta_k} \log \pi_(a_i | s) &= x \cdot \mathbf{1}[k = i] - \pi_\theta(a = k | s) \cdot x \\
    &= x \cdot [\mathbf{1}[k = i] - \pi_\theta(a = k | s)] \\
    &= \begin{cases}
        x \cdot [1 - \pi_\theta(a = k | s)] & \text{if } k = i \\
        x \cdot - \pi_\theta(a = k | s) & \text{otherwise }
    \end{cases} & \text{ Combining terms }
\end{aligned}

In [ ]:
np.outer(np.array([1,2,3]), np.array([2,3,4]))

In [ ]:
class SoftmaxPolicy:
    def __init__(self, input_dim, action_dim, lr=0.01):
        self.W = np.random.randn(action_dim, input_dim) * 0.1
        self.lr = lr

    def logits(self, input):
        return self.W @ input

    def probs(self, input):
        z = self.logits(input)
        z -= np.max(z) #subtract the max value of the logit output, to prevent exp(z) overflowing
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z)

    def sample(self, input):
        p = self.probs(input)
        a = np.random.choice(N_ACTIONS, p=p)
        return a, p

    def compute_gradient_of_log_pi(self, input, action):
        p = self.probs(input)
        
        ## p returns the probability of each N_ACTIONS
        ## For each row in `input`, we multiply by 
        grad = -np.outer(p, input)
        grad[action] += input        # add indicator term
        return grad

    def update(self, grad, G):
        self.W += self.lr * G * grad

In [ ]:
def reinforce(env, policy, gamma=0.99, episodes=500):
    for ep in range(episodes):
        obs = env.reset()
        traj = []  # store (obs, action, reward)

        done = False
        while not done:
            a, _ = policy.sample(obs)
            next_obs, r, done, info = env.step(a)
            traj.append((obs, a, r))
            obs = next_obs

        # Compute G_t backwards
        G = 0
        returns = []
        for (_, _, r) in reversed(traj):
            G = r + gamma * G
            returns.append(G)
        returns.reverse()

        # Update policy
        for (obs, a, r), Gt in zip(traj, returns):
            grad = policy.grad_logpi(obs, a)
            policy.update(grad, Gt)

env = Env()
policy = SoftmaxPolicy(env.input_dim, env.act_dim, lr=0.01)

reinforce(env, policy)